# 🌍 Multi-market fundamentals collector (Colab, robust)

Collects ~5y of **dated annual fundamentals** (all Piotroski inputs) per market via yfinance.

**Fixes for Colab:** uses **`curl_cffi` Chrome-impersonation** to get past Yahoo's datacenter-IP
block · a **fail-fast IP probe** (so you don't churn 40 min collecting nothing) · **checkpoints
to parquet + zip every 3 minutes** · **resumable** from an uploaded zip.

**Run:** Runtime → Run all. Download `/content/fundamentals.zip` anytime. If the probe says the
IP is blocked → Runtime → Restart runtime (fresh IP) → Run all again. Research/education only.

In [ ]:
# 1 · install + fetch the ticker universe (plain CSV, no LFS)
!pip -q install yfinance curl_cffi pandas pyarrow >/dev/null 2>&1
!wget -qO tickers.csv https://raw.githubusercontent.com/herrrickshaw/colab-market-fundamentals/main/tickers.csv
import pandas as pd
U = pd.read_csv('tickers.csv')
print('universe:', len(U), 'tickers,', U['market'].nunique(), 'markets')
print(U.groupby('market').size().to_dict())

In [ ]:
# 2 · (optional) RESUME — upload a previous fundamentals.zip to continue where you left off
import os, zipfile
os.makedirs('/content/fundamentals', exist_ok=True)
try:
    from google.colab import files
    up = files.upload()   # pick fundamentals.zip, or just cancel to start fresh
    for name in up:
        if name.endswith('.zip'):
            with zipfile.ZipFile(name) as z: z.extractall('/content/fundamentals')
            print('resumed from', name)
except Exception as e:
    print('no resume (starting fresh):', e)

In [ ]:
# 3 · collector: curl_cffi impersonation + rate limit + retry + 3-min parquet/zip checkpoint
import time, threading, random, glob, shutil
import yfinance as yf
from curl_cffi import requests as creq
from concurrent.futures import ThreadPoolExecutor, as_completed

OUT = '/content/fundamentals'; os.makedirs(OUT, exist_ok=True)
BARE, SKIP = {'IN'}, {'US'}
INCOME = {'net_income':['Net Income'], 'revenue':['Total Revenue'], 'gross_profit':['Gross Profit']}
BAL = {'total_assets':['Total Assets'], 'current_assets':['Current Assets'],
       'current_liabilities':['Current Liabilities'], 'long_term_debt':['Long Term Debt'],
       'shares':['Ordinary Shares Number','Share Issued'],
       'equity':['Stockholders Equity','Total Equity Gross Minority Interest']}
CASH = {'cfo':['Operating Cash Flow','Cash Flow From Continuing Operating Activities']}

_tl = threading.local()
def _sess():
    if not hasattr(_tl,'s'): _tl.s = creq.Session(impersonate='chrome')   # per-thread browser session
    return _tl.s
MIN_INTERVAL = 0.10; _rl = threading.Lock(); _last=[0.0]
def _throttle():
    with _rl:
        g = time.time()-_last[0]
        if g < MIN_INTERVAL: time.sleep(MIN_INTERVAL-g)
        _last[0] = time.time()
def _pick(df, labels):
    if df is None or df.empty: return {}
    for lab in labels:
        if lab in df.index:
            return {pd.Timestamp(c).normalize(): v for c,v in df.loc[lab].items() if pd.notna(v)}
    return {}
def fetch(tk, bare, retries=3):
    for a in range(retries):
        _throttle()
        try:
            t = yf.Ticker(tk, session=_sess())
            inc, bal, cash = t.income_stmt, t.balance_sheet, t.cashflow
            if (inc is None or inc.empty) and (bal is None or bal.empty): raise ValueError('empty')
            ser = {}
            for f,l in INCOME.items(): ser[f]=_pick(inc,l)
            for f,l in BAL.items():    ser[f]=_pick(bal,l)
            for f,l in CASH.items():   ser[f]=_pick(cash,l)
            fy = sorted({d for s in ser.values() for d in s})
            sym = tk.replace('.NS','').replace('.BO','').upper() if bare else tk.upper()
            out=[]
            for d in fy:
                row={'ticker':sym,'fy_end':d.strftime('%Y-%m-%d')}
                for f,s in ser.items():
                    if d in s: row[f]=float(s[d])
                row['filed']=(d+pd.Timedelta(days=90)).strftime('%Y-%m-%d')
                out.append(row)
            return out
        except Exception:
            time.sleep((2**a)+random.random())
    return []

# accumulator + periodic (3-min) flush to parquet + zip
_rows={}; _wlock=threading.Lock(); _stop=threading.Event()
def _load_existing():
    for p in glob.glob(f'{OUT}/*.parquet'):
        m=os.path.basename(p)[:-8]
        try: _rows[m]=pd.read_parquet(p).to_dict('records')
        except Exception: pass
def flush():
    with _wlock:
        for m,rr in _rows.items():
            if rr: pd.DataFrame(rr).to_parquet(f'{OUT}/{m}.parquet', index=False)
    shutil.make_archive('/content/fundamentals','zip',OUT)   # → /content/fundamentals.zip
def _flusher():
    while not _stop.wait(180):   # every 3 minutes
        flush(); print(f'  [checkpoint {time.strftime("%H:%M:%S")}] parquet+zip written')
def probe():
    ok=0
    for tk in ['AMZN','MSFT','VOLV-B.ST']:
        try:
            d=yf.Ticker(tk,session=_sess()).income_stmt
            if d is not None and not d.empty: ok+=1
        except Exception: pass
    return ok
def collect_market(m, tickers, workers=8):
    bare=m in BARE; _rows.setdefault(m,[])
    done={str(r['ticker']).upper() for r in _rows[m]}
    key=lambda tk:(tk.replace('.NS','').replace('.BO','').upper() if bare else tk.upper())
    todo=[t for t in tickers if key(t) not in done]
    print(f'  {m}: {len(tickers)} names, {len(done)} done, {len(todo)} to fetch')
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futs={ex.submit(fetch,tk,bare):tk for tk in todo}
        for fut in as_completed(futs):
            try:
                r=fut.result()
                if r:
                    with _wlock: _rows[m].extend(r)
            except Exception: pass
    print(f'  {m}: {len({r["ticker"] for r in _rows[m]})} names with data')
print('collector ready')

In [ ]:
# 4 · RUN — probe first (fail fast), then collect all markets with 3-min checkpoints
_load_existing()
p = probe()
print(f'IP probe: {p}/3 blue-chips returned data')
if p == 0:
    print('\n⚠️  This Colab IP is BLOCKED by Yahoo. Runtime → Restart runtime (new IP), then re-run.\n')
else:
    MARKETS = [m for m in sorted(U['market'].unique()) if m not in SKIP]
    print('collecting:', MARKETS)
    fl = threading.Thread(target=_flusher, daemon=True); fl.start()
    for m in MARKETS:
        tks = U[U['market']==m]['yf_ticker'].astype(str).tolist()
        collect_market(m, tks, workers=8)
        flush()                       # checkpoint after every market too
    _stop.set(); flush()
    print('\nALL DONE → /content/fundamentals.zip')
    for pth in sorted(glob.glob(f'{OUT}/*.parquet')):
        d = pd.read_parquet(pth, columns=['ticker'])
        print(f'  {os.path.basename(pth)[:-8]}: {d["ticker"].nunique()} tickers, {len(d)} rows')

In [ ]:
# 5 · download the zip (also written automatically every 3 min while running)
from google.colab import files
shutil.make_archive('/content/fundamentals','zip',OUT)
files.download('/content/fundamentals.zip')